#### Проект 1


In [ ]:
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


SEGMENT = "region_code <= 20"          # двадцать регионов из восьмидесяти девяти
WINDOW = ("2026-01-01", "2026-07-01")  # окно фактов: полугодие, правый конец не включён

print("подключились как:", q("select current_user as login").iloc[0, 0])

In [ ]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by p.payment_id
""")

sub_members = q(f"""
    select m.subscription_id, m.client_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")



In [ ]:
partners = q(f"""
    select *
    from prime.partners
""")

services = q(f"""
    select *
    from prime.services
""")

service_usage = q(f"""
    select * from prime.service_usage
    where {CLIENT_FILTER}
    """)




In [ ]:
transactions = q(f"""
    select *
    from prime.transactions
    where {CLIENT_FILTER}
""")

cashback = q(f"""
    select *
    from prime.cashback
""")

support_tickets = q(f"""
    select * from prime.support_tickets
    where {CLIENT_FILTER}
""")


In [ ]:
support_tickets.groupby('category')['client_id'].count()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sharing_tickets = support_tickets[support_tickets['category'] == 'доступ к сервису'].copy()

tickets_by_date = sharing_tickets.resample('D', on='created_at').size().reset_index(name='count')

# 4. Строим график
plt.figure(figsize=(12, 6))
sns.lineplot(data=tickets_by_date, x='created_at', y='count', marker='o')

plt.title('Количество тикетов категории "доступ к сервису" по датам')
plt.xlabel('Дата')
plt.ylabel('Количество тикетов')
plt.grid(True)
plt.xticks(rotation=45) 
plt.tight_layout()
plt.show()

#### Вопрос: с чем связан скачок тикетов в августе?

#### Сужаем отрезок


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

support_tickets['created_at'] = pd.to_datetime(support_tickets['created_at'])

sharing_tickets = support_tickets[support_tickets['category'] == 'доступ к сервису'].copy()

sharing_tickets = sharing_tickets[(sharing_tickets['created_at'] >= '2026-08-01') & (sharing_tickets['created_at'] < '2026-08-08')]

tickets_by_date = sharing_tickets.resample('D', on='created_at').size().reset_index(name='count')

plt.figure(figsize=(14, 6))
plt.plot(tickets_by_date['created_at'], tickets_by_date['count'], 
         marker='o', linestyle='-', color='teal', linewidth=2)

plt.title('Тикеты категории "доступ к сервису" за июнь–сентябрь', fontsize=14)
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Количество тикетов', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sharing_tickets = sharing_tickets[(sharing_tickets['created_at'] >= '2026-08-02') & (sharing_tickets['created_at'] < '2026-08-04')]
sharing_tickets.groupby('created_at').size().reset_index(name='count')

In [ ]:
tickets_by_category = support_tickets[(support_tickets['category'] == 'доступ к сервису')
 & (support_tickets['created_at'] >= '2026-08-02') & (support_tickets['created_at'] < '2026-08-04')]
problems_by_region = tickets_by_category.merge(clients[['client_id', 'region_code']], left_on='client_id', 
                                               right_on='client_id', how='left')
problems_by_region.groupby('region_code')['client_id'].size().sort_values(ascending=False).head(10)


#### Проблема системная, с регионами не связана

Выведем примеры тикетов

In [ ]:
support_tickets['text'][(support_tickets['category'] == 'доступ к сервису')
 & (support_tickets['created_at'] >= '2025-08-02') & (support_tickets['created_at'] < '2025-08-04')]

In [ ]:
service_usage['started_at'] = pd.to_datetime(service_usage['started_at'])
service_usage['ended_at'] = pd.to_datetime(service_usage['ended_at'])
# Смотрим динамику сессий в начале августа
august_usage = service_usage[
    (service_usage['started_at'] >= '2026-08-01') & 
    (service_usage['ended_at'] < '2026-09-01')
]

# Считаем количество сессий по дням
daily_sessions = august_usage.groupby(august_usage['started_at'].dt.date).size()

plt.figure(figsize=(10, 5))
daily_sessions.plot(kind='bar', color='skyblue')
plt.title('Количество сессий в сервисах (1-4 августа)')
plt.xlabel('Дата')
plt.ylabel('Кол-во сессий')
plt.xticks(rotation=45)
plt.show()

#### Дело не в сессиях, гипотеза о перегрузке сервера отметается

In [ ]:
tariffs

#### Тарифы в это время не менялись, гипотезу о смене тарифов отметаем

In [ ]:
problems_by_region = problems_by_region[['client_id', 'text', 'created_at', 'resolved_at']].drop_duplicates(subset='client_id', keep='first')
problems_by_region.sort_values(by='client_id', inplace=True)
problems_by_region

#### Попробуем сегментировать и посмотреть, из за каких долей произошел скачок

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = support_tickets[(support_tickets['category'] == 'доступ к сервису') & (support_tickets['created_at'].dt.date >= pd.Timestamp('2026-07-01').date())].copy()

def assign_group(text):
    if pd.isna(text):
        return 'Другое'
    if text.startswith('Купил расширенный тариф'):
        return 'А: Купил тариф'
    elif text.startswith('Не открывается сервис'):
        return 'B: Не открывается'
    elif text.startswith('Приложение не видит подписку после переустановки.'):
        return 'C: После переустановки'
    else:
        return 'Другое'

df['group'] = df['text'].apply(assign_group)

df_grouped = df.groupby([pd.Grouper(key='created_at', freq='D'), 'group']).size().reset_index(name='count')

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=df_grouped, 
    x='created_at', 
    y='count', 
    hue='group',       
    marker='o',
    linewidth=2
)

plt.title('Динамика тикетов категории "доступ к сервису" по группам', fontsize=14)
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Количество тикетов', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.legend(title='Группа жалоб', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels

In [ ]:
import statsmodels
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose, STL
daily_total = (df
    .groupby(pd.Grouper(key='created_at', freq='D'))
    .size()
    .rename('count'))
daily_total = daily_total.asfreq('D').fillna(0)

print(f"Длина ряда: {len(daily_total)} дней")
print(f"Период: {daily_total.index.min()} — {daily_total.index.max()}")
decomp = seasonal_decompose(daily_total, model='additive', period=7)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

daily_total.plot(ax=axes[0], color='black')
axes[0].set_title('Исходный ряд (Observed)', fontsize=12)
axes[0].set_ylabel('Тикеты')

decomp.trend.plot(ax=axes[1], color='blue')
axes[1].set_title('Тренд (Trend)', fontsize=12)
axes[1].set_ylabel('Тикеты')

decomp.seasonal.plot(ax=axes[2], color='green')
axes[2].set_title('Сезонность (Seasonal, period=7)', fontsize=12)
axes[2].set_ylabel('Отклонение')

decomp.resid.plot(ax=axes[3], color='red')
axes[3].set_title('Остаток (Residual)', fontsize=12)
axes[3].set_ylabel('Отклонение')
axes[3].set_xlabel('Дата')

plt.tight_layout()
plt.show()

print("\n Метрики разложения")
print(f"Средний тренд: {decomp.trend.mean():.2f}")
print(f"Амплитуда сезонности: {decomp.seasonal.max() - decomp.seasonal.min():.2f}")
print(f"Стд. остатка: {decomp.resid.std():.2f}")

var_total = daily_total.var()
var_trend = decomp.trend.var()
var_seasonal = decomp.seasonal.var()
var_resid = decomp.resid.var()

print(f"\nДоля дисперсии:")
print(f"  Тренд: {var_trend / var_total:.2%}")
print(f"  Сезонность: {var_seasonal / var_total:.2%}")
print(f"  Остаток: {var_resid / var_total:.2%}")

In [ ]:

df_grouped = df.groupby([pd.Grouper(key='created_at', freq='D'), 'group']).size().reset_index(name='count')
df_grouped['date_str'] = df_grouped['created_at'].dt.strftime('%Y-%m-%d')

pivot_df = df_grouped.pivot(index='date_str', columns='group', values='count').fillna(0)
pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(14, 6),
    colormap='Set2',
    edgecolor='white'
)

plt.title('Структура тикетов "доступ к сервису" по группам', fontsize=14)
plt.xlabel('Дата', fontsize=12)
plt.ylabel('Количество тикетов', fontsize=12)
plt.legend(title='Группа жалоб', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

#### Видим, что в скачке наблюдается значительный вклад следующих долей: не открывается сервис из за неактивности подписки и проблема с переходом на расширенный тариф

In [ ]:
problem_clients = problems_by_region

groupa = problem_clients[problem_clients['text'].str.startswith('Купил расширенный тариф')]
groupb = problem_clients[problem_clients['text'].str.startswith('Не открывается сервис')]
#groupc = problem_clients[problem_clients['text'].str.startswith('Приложение не видит подписку после переустановки.')]
problema = groupa.merge(subs, left_on='client_id', right_on='owner_client_id', how='left')[['client_id', 'text']].drop_duplicates()
groupa.merge(subs, left_on='client_id', right_on='owner_client_id', how='left')[['client_id', 'text', 'subscription_id', 'created_at', 'tariff_id', 'started_at', 'ended_at', 'cancel_reason']]
#groupa.merge(sub_members, left_on='client_id', right_on='client_id', how='left')[['client_id', 'text', 'subscription_id', 'created_at']]
#groupc.merge(sub_members, left_on='client_id', right_on='client_id', how='left')#[['client_id', 'text', 'created_at', 'subscription_id','joined_at']]



У всех из группы А(те, кто купил расширенную подписку), кроме 2х человек,не сменилось на расширенную подписку. 

Те, у кого сменилось на расширенную: 

06f73935-4c43-4581-945b-a3d2636bd939, ----- (вероятные технические проблемы)

3652f35b-c226-4447-bc1d-883fac5a6e3f ------- (проблема с оплатой)

Подозрительные записи, которых нет в subs:

430e748f-8481-472c-9b81-408688c20ff0,  

642016ab-16a1-47e0-96ba-f67825a9bd57, 

 81ae4b47-cf59-4d5f-b27b-f533102f810f

Записи со значащимся тарифом 1:

2aa2f8d6-b88d-4f16-88d4-1f2eba3b6e0d

4348e595-96e9-4bce-84f9-c7a7728bbab5

bfbde1e8-7f3a-4566-9f57-4ee0019152e4

#### Жалующиеся на отсутствие доступа к каршерингу в большинстве своем продолжают платить за базовую подписку и не имеют записей об изменениях тарифа. Также некоторые не имеют записи в subs, что подводит к предположению о манипуляции с попытками получения расширенной подписки без оплаты.

In [ ]:
result = (problema
    .merge(transactions, on='client_id', how='left')
    .loc[lambda df: df["occurred_at"] > "2026-08-01",
         ['client_id', 'transaction_id', 'amount', 'status', 'occurred_at', 'service_id']])
result.merge(services, left_on='service_id', right_on='service_id', how='left')

In [ ]:
sub_members[sub_members['client_id'] == '430e748f-8481-472c-9b81-408688c20ff0']
subs[subs['owner_client_id'] == '430e748f-8481-472c-9b81-408688c20ff0']

In [ ]:
test1 = transactions.merge(services, on = 'service_id', how='left')[['client_id', 'transaction_id', 'amount', 'status', 'occurred_at', 'service_id', 'available_from_tariff']]
test2 = test1[test1['available_from_tariff'] >= 1].merge(subs, left_on='client_id', right_on='owner_client_id', how='left')
not_in_subs = test2[test2["owner_client_id"].isna()]['client_id'] #это от subs, ща посмотрим по members 

not_in_subs_nor_members = not_in_subs[~not_in_subs.isin(sub_members["client_id"])]

not_in_subs_nor_members.nunique()

#### А вот и клиенты группы А, которые делают транзации по подписке, но которых нет ни в subs, ни в sub_members

In [ ]:
problemb = groupb.merge(subs, left_on='client_id', right_on='owner_client_id', how='left')[['client_id', 'text', 'subscription_id', 'created_at', 'resolved_at', 'tariff_id', 'started_at', 'ended_at', 'cancel_reason']]
problemb 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pay['paid_at'] = pd.to_datetime(pay['paid_at'])

pay_filtered = pay[
    (pay['paid_at'] >= '2026-07-01') & 
    (pay['paid_at'] < '2026-09-01')
].copy()

pay_filtered['date_str'] = pay_filtered['paid_at'].dt.strftime('%Y-%m-%d')


pay_grouped = pay_filtered.groupby(['date_str', 'status']).size().reset_index(name='count')
pivot_df = pay_grouped.pivot(index='date_str', columns='status', values='count').fillna(0)
status_order = ['success', 'failed', 'refunded']
pivot_df = pivot_df[[s for s in status_order if s in pivot_df.columns]]

colors = {'success': '#4CAF50', 'failed': '#F44336', 'refunded': '#FF9800'}
fig, ax = plt.subplots(figsize=(16, 6))

bottom = np.zeros(len(pivot_df))
for col in pivot_df.columns:
    ax.bar(pivot_df.index, pivot_df[col], bottom=bottom, 
           label=col, color=colors.get(col, '#999999'), edgecolor='white')
    bottom += pivot_df[col].values

ax.set_title('Платежи по дням (июль-август) с разбивкой по статусам', fontsize=14)
ax.set_xlabel('Дата', fontsize=12)
ax.set_ylabel('Количество платежей', fontsize=12)
ax.legend(title='Статус', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=90, fontsize=8)  # поворот на 90, чтобы влезли все даты
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

#### Значительных падений по таблице pay не замечено 

In [ ]:
groupb.merge(sub_members, left_on='client_id', right_on='client_id', how='inner')[['client_id', 'text', 'created_at', 'subscription_id']]


#### Они не участвуют в других подписках, только в своих

In [ ]:
problemb[['client_id', 'created_at', 'resolved_at']].merge(service_usage[(service_usage['started_at'].dt.date >= pd.Timestamp('2026-08-01').date()) & (service_usage['started_at'].dt.date <= pd.Timestamp('2026-08-05').date())], on='client_id', how='left')

#### По группе Б - действительно возникли технические неполадки на стороне сервиса, которые быстро разрешились

#### Собираем метрики

In [ ]:
#  метрики масштаба аномалии 

daily = (df
    .groupby(df["created_at"].dt.date)
    .size()
    .rename("count"))

ANOMALY_START = pd.Timestamp("2026-08-03").date()
ANOMALY_END   = pd.Timestamp("2026-08-04").date()

peak = daily.loc[ANOMALY_START:ANOMALY_END].max()
background = daily.drop(index=daily.loc[ANOMALY_START:ANOMALY_END].index).mean()
threshold = daily.drop(index=daily.loc[ANOMALY_START:ANOMALY_END].index).quantile(0.95)
anomaly_days = daily[daily > threshold]

metrics_scale = pd.DataFrame({
    "Показатель": [
        "Пиковое значение жалоб за день",
        "Среднее значение жалоб за день (фон)",
        "Отношение пик / фон",
        "Амплитуда скачка (пик − фон)",
        "Порог аномалии (95-й перцентиль фона)",
        "Длительность аномалии (дней)",
        "Доля скачка в общем числе жалоб, %",
    ],
    "Значение": [
        int(peak),
        round(background, 2),
        round(peak / background, 2),
        int(peak - background),
        round(threshold, 2),
        len(anomaly_days),
        round(daily.loc[ANOMALY_START:ANOMALY_END].sum() / daily.sum() * 100, 2),
    ]
})
metrics_scale

In [ ]:
#  метрики клиентов без подписки и участия 

tickets_anomaly = df[
    (df["created_at"].dt.date >= ANOMALY_START) &
    (df["created_at"].dt.date <= ANOMALY_END)
]

clients_in_tickets = set(tickets_anomaly["client_id"].dropna())
owners = set(subs["owner_client_id"].dropna())
members = set(sub_members["client_id"].dropna())
phantom_clients = clients_in_tickets - owners - members

phantom_tx = transactions[transactions["client_id"].isin(phantom_clients)]
phantom_revenue = phantom_tx.loc[phantom_tx["status"] == "success", "amount"].sum()

metrics_phantom = pd.DataFrame({
    "Показатель": [
        "Всего клиентов в жалобах 3–4 августа",
        "Из них нет в subs и sub_members",
        "Доля фантомных клиентов, %",
        "Успешных транзакций у фантомных клиентов",
        "Выручка от фантомных клиентов, ₽",
        "Средний чек фантомного клиента, ₽",
    ],
    "Значение": [
        len(clients_in_tickets),
        len(phantom_clients),
        round(len(phantom_clients) / len(clients_in_tickets) * 100, 2),
        len(phantom_tx[phantom_tx["status"] == "success"]),
        round(phantom_revenue, 2),
        round(phantom_revenue / max(len(phantom_tx[phantom_tx["status"] == "success"]), 1), 2),
    ]
})
metrics_phantom